# 01 — Discovery y Perfilado de Datos

Para cada tabla revisamos:
- Volumen y estructura (filas, columnas, tipos)
- Nulos por columna
- Duplicados (filas completas y por clave primaria)
- Cardinalidad de columnas clave
- Llaves huérfanas (FK que no existen en la tabla referenciada)
- Outliers / valores fuera de rango en columnas numéricas y de fecha


In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 150)

def get_engine():
    host = os.environ.get("WAREHOUSE_HOST", "postgres-warehouse")
    port = os.environ.get("WAREHOUSE_PORT", "5432")
    db = os.environ.get("WAREHOUSE_DB", "warehouse")
    user = os.environ.get("WAREHOUSE_USER", "rodrick")
    password = os.environ.get("WAREHOUSE_PASSWORD", "rodrick123")
    url = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}"
    return create_engine(url)

engine = get_engine()
with engine.connect() as conn:
    conn.execute(text("SELECT 1"))
print("Conexión OK.")


Conexión OK.


In [50]:
tables_list = pd.read_sql("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'bronze'
      AND table_name != '_ingestion_log'
    ORDER BY table_name;
""", engine)["table_name"].tolist()

for t in tables_list:
    df_sample = pd.read_sql(f'SELECT * FROM bronze."{t}" LIMIT 5', engine)
    n_rows = pd.read_sql(f'SELECT COUNT(*) AS n FROM bronze."{t}"', engine)["n"][0]
    print(f"\n{'='*100}")
    print(f"  {t}   ({n_rows} filas totales, {df_sample.shape[1]} columnas)")
    print('='*100)
    display(df_sample)


  billing_customers   (10000 filas totales, 11 columnas)


,customer_id,external_ref,first_name,last_name,email,country,created_at,segment,_source_file,_source_domain,_ingested_at
0,CUS-0000001,STU-0000001,Carlos,Contreras,carlos.contreras7090@example.com,CL,2021-07-14 19:53:33,smb,customers.csv,billing,2026-07-17T14:04:44.108076+00:00
1,CUS-0000002,STU-0000002,Maria,Sandoval,maria.sandoval1480@lake.local,MX,2019-02-01 06:07:23,retail,customers.csv,billing,2026-07-17T14:04:44.108076+00:00
2,CUS-0000003,STU-0000003,Ignacio,Torres,ignacio.torres8864@example.com,CL,2018-06-03 21:02:27,retail,customers.csv,billing,2026-07-17T14:04:44.108076+00:00
3,CUS-0000004,STU-0000004,Juan,Sandoval,juan.sandoval3659@demo.io,MX,2019-08-29 12:43:31,smb,customers.csv,billing,2026-07-17T14:04:44.108076+00:00
4,CUS-0000005,STU-0000005,Cristobal,Reyes,cristobal.reyes1471@mail.test,CL,2020-08-06 11:27:12,retail,customers.csv,billing,2026-07-17T14:04:44.108076+00:00



  billing_invoice_items   (150000 filas totales, 9 columnas)


,invoice_item_id,quantity,unit_price,line_total,invoice_id,product_id,_source_file,_source_domain,_ingested_at
0,INI-000000001,7,64.65,452.55000000000007,INV-00029386,PRD-00117,invoice_items.csv,billing,2026-07-17T14:04:45.009129+00:00
1,INI-000000002,1,20.39,20.39,INV-00045051,PRD-00142,invoice_items.csv,billing,2026-07-17T14:04:45.009129+00:00
2,INI-000000003,7,36.59,256.13,INV-00029813,PRD-00117,invoice_items.csv,billing,2026-07-17T14:04:45.009129+00:00
3,INI-000000004,8,49.38,395.04,INV-00032937,PRD-00110,invoice_items.csv,billing,2026-07-17T14:04:45.009129+00:00
4,INI-000000005,3,24.91,74.73,INV-00032440,PRD-00140,invoice_items.csv,billing,2026-07-17T14:04:45.009129+00:00



  billing_invoices   (50000 filas totales, 10 columnas)


...

,professor_id,first_name,last_name,email,department,hired_at,_source_file,_source_domain,_ingested_at
0,PRF-00001,Matias,Sandoval,matias.sandoval4330@lake.local,cs,2010-11-10,professors.csv,university,2026-07-17T14:04:43.741913+00:00
1,PRF-00002,Gabriel,Arancibia,gabriel.arancibia7939@demo.io,cs,2021-05-27,professors.csv,university,2026-07-17T14:04:43.741913+00:00
2,PRF-00003,Rodrigo,Sandoval,rodrigo.sandoval265@demo.io,cs,2022-06-20,professors.csv,university,2026-07-17T14:04:43.741913+00:00
3,PRF-00004,Matias,Fuentes,matias.fuentes5187@mail.test,math,2009-09-01,professors.csv,university,2026-07-17T14:04:43.741913+00:00
4,PRF-00005,Lucia,Gutierrez,lucia.gutierrez6256@demo.io,cs,2012-03-23,professors.csv,university,2026-07-17T14:04:43.741913+00:00



  university_semesters   (8 filas totales, 9 columnas)


,semester_id,code,year,half,start_date,end_date,_source_file,_source_domain,_ingested_at
0,SEM-001,2022-1,2022,1,2022-03-01,2022-07-15,semesters.csv,university,2026-07-17T14:04:43.781445+00:00
1,SEM-002,2022-2,2022,2,2022-08-01,2022-12-15,semesters.csv,university,2026-07-17T14:04:43.781445+00:00
2,SEM-003,2023-1,2023,1,2023-03-01,2023-07-15,semesters.csv,university,2026-07-17T14:04:43.781445+00:00
3,SEM-004,2023-2,2023,2,2023-08-01,2023-12-15,semesters.csv,university,2026-07-17T14:04:43.781445+00:00
4,SEM-005,2024-1,2024,1,2024-03-01,2024-07-15,semesters.csv,university,2026-07-17T14:04:43.781445+00:00



  university_students   (5000 filas totales, 10 columnas)


,student_id,first_name,last_name,email,birth_date,enrolled_at,country,_source_file,_source_domain,_ingested_at
0,STU-0000001,Martina,Diaz,martina.diaz5727@lake.local,2000-12-21,2019-10-01,US,students.csv,university,2026-07-17T14:04:43.832719+00:00
1,STU-0000002,Manuel,Torres,manuel.torres5619@mail.test,2004-08-10,2025-03-20,CL,students.csv,university,2026-07-17T14:04:43.832719+00:00
2,STU-0000003,Maximiliano,Martinez,maximiliano.martinez7688@demo.io,2007-08-10,2022-10-19,PE,students.csv,university,2026-07-17T14:04:43.832719+00:00
3,STU-0000004,Magdalena,Vasquez,magdalena.vasquez8686@example.com,2002-05-13,2020-09-23,CL,students.csv,university,2026-07-17T14:04:43.832719+00:00
4,STU-0000005,Luis,Rivera,luis.rivera9349@lake.local,2005-12-18,2025-07-28,CL,students.csv,university,2026-07-17T14:04:43.832719+00:00


In [2]:
tables = pd.read_sql("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'bronze'
    ORDER BY table_name;
""", engine)
tables


,table_name
0,_ingestion_log
1,billing_customers
2,billing_invoice_items
3,billing_invoices
4,billing_payments
5,billing_products
6,billing_subscriptions
7,crm_accounts
8,crm_activities
9,crm_contacts


## Funciones de perfilado


In [3]:
def load_table(table_name, schema="bronze"):
    return pd.read_sql(f'SELECT * FROM {schema}."{table_name}"', engine)


def null_report(df, table_name):
    """Cantidad y % de nulos/vacíos por columna."""
    nulls = df.isna().sum()
    empties = (df == "").sum(numeric_only=False)
    total_missing = nulls + empties
    report = pd.DataFrame({
        "nulls": nulls,
        "empty_strings": empties,
        "total_missing": total_missing,
        "pct_missing": (total_missing / len(df) * 100).round(2)
    })
    report = report[report["total_missing"] > 0].sort_values("pct_missing", ascending=False)
    print(f"[{table_name}] columnas con datos faltantes:")
    return report


def duplicate_report(df, table_name, key_cols=None):
    """Duplicados por fila completa y, opcionalmente, por columna(s) clave."""
    full_dupes = df.duplicated().sum()
    print(f"[{table_name}] filas completamente duplicadas: {full_dupes}")
    if key_cols:
        key_dupes = df.duplicated(subset=key_cols).sum()
        print(f"[{table_name}] duplicados por clave {key_cols}: {key_dupes}")
    return full_dupes


def cardinality_report(df, table_name, cols):
    """Valores únicos por columna, útil para detectar columnas casi-constantes
    o con más variedad de la esperada."""
    report = pd.DataFrame({
        "n_unique": [df[c].nunique() for c in cols],
        "n_total": len(df),
    }, index=cols)
    report["pct_unique"] = (report["n_unique"] / report["n_total"] * 100).round(2)
    print(f"[{table_name}] cardinalidad:")
    return report


def orphan_keys(df_child, fk_col, df_parent, pk_col, child_name, parent_name):
    """Filas del hijo cuya FK no existe en el padre (llaves huérfanas)."""
    child_ids = set(df_child[fk_col].dropna())
    parent_ids = set(df_parent[pk_col].dropna())
    orphans = child_ids - parent_ids
    print(f"[{child_name}.{fk_col} -> {parent_name}.{pk_col}] "
          f"huérfanos: {len(orphans)} de {len(child_ids)} valores únicos")
    return orphans


---
## Dominio: `university`


In [4]:
semesters   = load_table("university_semesters")
professors  = load_table("university_professors")
students    = load_table("university_students")
courses     = load_table("university_courses")
enrollments = load_table("university_enrollments")
grades      = load_table("university_grades")

students.head()


,student_id,first_name,last_name,email,birth_date,enrolled_at,country,_source_file,_source_domain,_ingested_at
0,STU-0000001,Martina,Diaz,martina.diaz5727@lake.local,2000-12-21,2019-10-01,US,students.csv,university,2026-07-17T14:04:43.832719+00:00
1,STU-0000002,Manuel,Torres,manuel.torres5619@mail.test,2004-08-10,2025-03-20,CL,students.csv,university,2026-07-17T14:04:43.832719+00:00
2,STU-0000003,Maximiliano,Martinez,maximiliano.martinez7688@demo.io,2007-08-10,2022-10-19,PE,students.csv,university,2026-07-17T14:04:43.832719+00:00
3,STU-0000004,Magdalena,Vasquez,magdalena.vasquez8686@example.com,2002-05-13,2020-09-23,CL,students.csv,university,2026-07-17T14:04:43.832719+00:00
4,STU-0000005,Luis,Rivera,luis.rivera9349@lake.local,2005-12-18,2025-07-28,CL,students.csv,university,2026-07-17T14:04:43.832719+00:00


In [5]:
for name, df in [("semesters", semesters), ("professors", professors), ("students", students),
                  ("courses", courses), ("enrollments", enrollments), ("grades", grades)]:
    print(f"{name:15s} shape={df.shape}")


semesters       shape=(8, 9)
professors      shape=(200, 9)
students        shape=(5000, 10)
courses         shape=(300, 9)
enrollments     shape=(25000, 9)
grades          shape=(60000, 9)


In [6]:
null_report(students, "students")


[students] columnas con datos faltantes:


,nulls,empty_strings,total_missing,pct_missing


In [7]:
null_report(enrollments, "enrollments")


[enrollments] columnas con datos faltantes:


,nulls,empty_strings,total_missing,pct_missing


In [8]:
null_report(grades, "grades")


[grades] columnas con datos faltantes:


,nulls,empty_strings,total_missing,pct_missing


In [9]:
duplicate_report(students, "students", key_cols=["student_id"])
duplicate_report(enrollments, "enrollments", key_cols=["enrollment_id"])
duplicate_report(grades, "grades", key_cols=["enrollment_id"])


[students] filas completamente duplicadas: 0
[students] duplicados por clave ['student_id']: 0
[enrollments] filas completamente duplicadas: 0
[enrollments] duplicados por clave ['enrollment_id']: 0
[grades] filas completamente duplicadas: 0
[grades] duplicados por clave ['enrollment_id']: 37214


0

In [10]:
orphan_keys(enrollments, "student_id", students, "student_id", "enrollments", "students")
orphan_keys(enrollments, "course_id", courses, "course_id", "enrollments", "courses")
orphan_keys(courses, "professor_id", professors, "professor_id", "courses", "professors")
orphan_keys(grades, "enrollment_id", enrollments, "enrollment_id", "grades", "enrollments")


[enrollments.student_id -> students.student_id] huérfanos: 0 de 4962 valores únicos
[enrollments.course_id -> courses.course_id] huérfanos: 0 de 300 valores únicos
[courses.professor_id -> professors.professor_id] huérfanos: 0 de 150 valores únicos
[grades.enrollment_id -> enrollments.enrollment_id] huérfanos: 0 de 22786 valores únicos


set()

In [14]:
grades_numeric = pd.to_numeric(grades["score"], errors="coerce")  # ajustar nombre de columna
print("Valores no convertibles a número:", grades_numeric.isna().sum() - grades["score"].isna().sum())
print(grades_numeric.describe())


Valores no convertibles a número: 0
count    60000.000000
mean        74.884157
std         11.818063
min         24.530000
25%         66.840000
50%         74.980000
75%         83.100000
max        100.000000
Name: score, dtype: float64


### Hallazgos — `university`

- Nulos:
- Duplicados:
- Llaves huérfanas:
- Outliers / rangos inválidos:


---
## Dominio: `billing`

Tablas: `customers`, `products`, `subscriptions`, `invoices`, `invoice_items`, `payments`.

- `subscriptions.customer_id` → `customers.customer_id`
- `subscriptions.product_id` → `products.product_id`
- `invoices.customer_id` → `customers.customer_id`
- `invoice_items.invoice_id` → `invoices.invoice_id`
- `payments.invoice_id` → `invoices.invoice_id`


In [15]:
customers      = load_table("billing_customers")
products       = load_table("billing_products")
subscriptions  = load_table("billing_subscriptions")
invoices       = load_table("billing_invoices")
invoice_items  = load_table("billing_invoice_items")
payments       = load_table("billing_payments")

invoices.head()


,invoice_id,issued_at,due_at,total,status,currency,customer_id,_source_file,_source_domain,_ingested_at
0,INV-00000001,2025-08-18,2025-09-05,107.64,paid,CLP,CUS-0000003,invoices.csv,billing,2026-07-17T14:04:51.212627+00:00
1,INV-00000002,2024-05-14,2024-06-04,70.06,paid,EUR,CUS-0000281,invoices.csv,billing,2026-07-17T14:04:51.212627+00:00
2,INV-00000003,2025-07-23,2025-08-31,35.12,paid,PEN,CUS-0001990,invoices.csv,billing,2026-07-17T14:04:51.212627+00:00
3,INV-00000004,2023-06-09,2023-06-18,16.19,paid,COP,CUS-0006989,invoices.csv,billing,2026-07-17T14:04:51.212627+00:00
4,INV-00000005,2023-07-18,2023-09-01,27.62,paid,EUR,CUS-0000366,invoices.csv,billing,2026-07-17T14:04:51.212627+00:00


In [16]:
for name, df in [("customers", customers), ("products", products), ("subscriptions", subscriptions),
                  ("invoices", invoices), ("invoice_items", invoice_items), ("payments", payments)]:
    print(f"{name:15s} shape={df.shape}")


customers       shape=(10000, 11)
products        shape=(200, 9)
subscriptions   shape=(15000, 9)
invoices        shape=(50000, 10)
invoice_items   shape=(150000, 9)
payments        shape=(80000, 8)


In [17]:
null_report(customers, "customers")


[customers] columnas con datos faltantes:


,nulls,empty_strings,total_missing,pct_missing
external_ref,5000,0,5000,50.0


In [18]:
null_report(invoices, "invoices")


[invoices] columnas con datos faltantes:


,nulls,empty_strings,total_missing,pct_missing


In [19]:
null_report(payments, "payments")


[payments] columnas con datos faltantes:


,nulls,empty_strings,total_missing,pct_missing


In [20]:
duplicate_report(customers, "customers", key_cols=["customer_id"])
duplicate_report(invoices, "invoices", key_cols=["invoice_id"])
duplicate_report(payments, "payments", key_cols=["payment_id"])


[customers] filas completamente duplicadas: 0
[customers] duplicados por clave ['customer_id']: 0
[invoices] filas completamente duplicadas: 0
[invoices] duplicados por clave ['invoice_id']: 0
[payments] filas completamente duplicadas: 0
[payments] duplicados por clave ['payment_id']: 0


0

In [21]:
orphan_keys(subscriptions, "customer_id", customers, "customer_id", "subscriptions", "customers")
orphan_keys(subscriptions, "product_id", products, "product_id", "subscriptions", "products")
orphan_keys(invoices, "customer_id", customers, "customer_id", "invoices", "customers")
orphan_keys(invoice_items, "invoice_id", invoices, "invoice_id", "invoice_items", "invoices")
orphan_keys(payments, "invoice_id", invoices, "invoice_id", "payments", "invoices")


[subscriptions.customer_id -> customers.customer_id] huérfanos: 0 de 7776 valores únicos
[subscriptions.product_id -> products.product_id] huérfanos: 0 de 200 valores únicos
[invoices.customer_id -> customers.customer_id] huérfanos: 0 de 9933 valores únicos
[invoice_items.invoice_id -> invoices.invoice_id] huérfanos: 0 de 47498 valores únicos
[payments.invoice_id -> invoices.invoice_id] huérfanos: 0 de 31433 valores únicos


set()

In [22]:
amount_col = "amount"  # ajustar
amounts = pd.to_numeric(payments[amount_col], errors="coerce")
print(amounts.describe())
print("Negativos:", (amounts < 0).sum())
print("Ceros:", (amounts == 0).sum())


count    80000.000000
mean        81.171059
std        103.606248
min          1.100000
25%         25.000000
50%         49.750000
75%         97.882500
max       3987.090000
Name: amount, dtype: float64
Negativos: 0
Ceros: 0


### Hallazgos — `billing`

- Nulos:
- Duplicados:
- Llaves huérfanas:
- Outliers / montos inválidos:


---
## Dominio: `crm`

Tablas: `accounts`, `contacts`, `leads`, `opportunities`, `opportunity_contacts`, `activities`.

- `contacts.account_id` → `accounts.account_id`
- `opportunities.account_id` → `accounts.account_id`
- `opportunity_contacts.opportunity_id` → `opportunities.opportunity_id`
- `opportunity_contacts.contact_id` → `contacts.contact_id`
- `activities.contact_id` → `contacts.contact_id` (verificar si también referencia opportunity_id)


In [23]:
accounts              = load_table("crm_accounts")
contacts              = load_table("crm_contacts")
leads                 = load_table("crm_leads")
opportunities         = load_table("crm_opportunities")
opportunity_contacts  = load_table("crm_opportunity_contacts")
activities            = load_table("crm_activities")

opportunities.head()


,opportunity_id,name,stage,amount,close_date,created_at,account_id,_source_file,_source_domain,_ingested_at
0,OPP-0000001,Deal 0000001,won,5746.66,2023-11-30,2022-03-19 16:45:26,ACC-0002022,opportunities.csv,crm,2026-07-17T14:04:59.117188+00:00
1,OPP-0000002,Deal 0000002,negotiation,24990.51,2023-09-28,2025-05-25 03:05:15,ACC-0000582,opportunities.csv,crm,2026-07-17T14:04:59.117188+00:00
2,OPP-0000003,Deal 0000003,qualification,11892.82,2024-07-02,2023-12-23 16:49:50,ACC-0002591,opportunities.csv,crm,2026-07-17T14:04:59.117188+00:00
3,OPP-0000004,Deal 0000004,proposal,31211.68,2023-09-19,2023-05-19 04:41:38,ACC-0004529,opportunities.csv,crm,2026-07-17T14:04:59.117188+00:00
4,OPP-0000005,Deal 0000005,proposal,27434.07,2023-08-30,2024-11-30 06:25:22,ACC-0003484,opportunities.csv,crm,2026-07-17T14:04:59.117188+00:00


In [24]:
for name, df in [("accounts", accounts), ("contacts", contacts), ("leads", leads),
                  ("opportunities", opportunities), ("opportunity_contacts", opportunity_contacts),
                  ("activities", activities)]:
    print(f"{name:15s} shape={df.shape}")


accounts        shape=(5000, 10)
contacts        shape=(15000, 11)
leads           shape=(2000, 11)
opportunities   shape=(3000, 10)
opportunity_contacts shape=(6000, 6)
activities      shape=(20000, 9)


In [25]:
null_report(accounts, "accounts")


[accounts] columnas con datos faltantes:


,nulls,empty_strings,total_missing,pct_missing


In [26]:
null_report(opportunities, "opportunities")


[opportunities] columnas con datos faltantes:


,nulls,empty_strings,total_missing,pct_missing


In [27]:
null_report(activities, "activities")


[activities] columnas con datos faltantes:


,nulls,empty_strings,total_missing,pct_missing
opportunity_id,9985,0,9985,49.92
contact_id,5976,0,5976,29.88


In [28]:
duplicate_report(accounts, "accounts", key_cols=["account_id"])
duplicate_report(contacts, "contacts", key_cols=["contact_id"])
duplicate_report(opportunities, "opportunities", key_cols=["opportunity_id"])


[accounts] filas completamente duplicadas: 0
[accounts] duplicados por clave ['account_id']: 0
[contacts] filas completamente duplicadas: 0
[contacts] duplicados por clave ['contact_id']: 0
[opportunities] filas completamente duplicadas: 0
[opportunities] duplicados por clave ['opportunity_id']: 0


0

In [29]:
orphan_keys(contacts, "account_id", accounts, "account_id", "contacts", "accounts")
orphan_keys(opportunities, "account_id", accounts, "account_id", "opportunities", "accounts")
orphan_keys(opportunity_contacts, "opportunity_id", opportunities, "opportunity_id",
            "opportunity_contacts", "opportunities")
orphan_keys(opportunity_contacts, "contact_id", contacts, "contact_id",
            "opportunity_contacts", "contacts")


[contacts.account_id -> accounts.account_id] huérfanos: 0 de 4749 valores únicos
[opportunities.account_id -> accounts.account_id] huérfanos: 0 de 2262 valores únicos
[opportunity_contacts.opportunity_id -> opportunities.opportunity_id] huérfanos: 0 de 2586 valores únicos
[opportunity_contacts.contact_id -> contacts.contact_id] huérfanos: 0 de 4955 valores únicos


set()

In [34]:
print(opportunities["stage"].value_counts(dropna=False)) 

stage
prospect         621
qualification    611
proposal         569
won              476
negotiation      420
lost             303
Name: count, dtype: int64


### Hallazgos — `crm`

- Nulos:
- Duplicados:
- Llaves huérfanas:
- Inconsistencias categóricas (mayúsculas/minúsculas, variantes de texto):


---
## Auditoría de tipos y formatos (fechas, montos, texto)



In [35]:
import re

def format_signature(series, max_samples=2000):
    """Convierte cada valor a una 'firma' de formato (dígitos->D, letras->A),
    para detectar mezclas de formato -- por ejemplo fechas en distintos patrones
    dentro de la misma columna."""
    def sig(v):
        s = str(v)
        s = re.sub(r'[A-Za-z]', 'A', s)
        s = re.sub(r'\d', 'D', s)
        return s
    sample = series.dropna()
    sample = sample[sample != ""]
    if len(sample) > max_samples:
        sample = sample.sample(max_samples, random_state=42)
    return sample.map(sig).value_counts()


def audit_column_types(df, table_name):
    """Revisa cada columna de la tabla y reporta inconsistencias de formato
    según lo que su nombre sugiere que debería contener."""
    print(f"\n=== {table_name} ===")
    found_issue = False
    for col in df.columns:
        if col.startswith("_source") or col == "_ingested_at":
            continue
        s = df[col]
        non_null = s[(s.notna()) & (s != "")]
        if len(non_null) == 0:
            continue
        lc = col.lower()

        # Columnas de fecha
        if "date" in lc or lc.endswith("_at") or lc.endswith("_on"):
            parsed = pd.to_datetime(non_null, errors="coerce")
            n_fail = parsed.isna().sum()
            sig = format_signature(non_null)
            if n_fail > 0 or len(sig) > 1:
                found_issue = True
                print(f"  [FECHA] {col}: {n_fail} valores no parseables | {len(sig)} formato(s) distinto(s) detectado(s)")
                print(sig.head(5).to_string())

        # Columnas numéricas / montos
        elif any(k in lc for k in ["amount", "price", "total", "rate", "grade",
                                     "score", "balance", "qty", "quantity", "cost"]):
            cleaned = non_null.astype(str).str.replace(r'[,$\s]', '', regex=True)
            numeric = pd.to_numeric(cleaned, errors="coerce")
            n_fail = numeric.isna().sum()
            if n_fail > 0:
                found_issue = True
                print(f"  [NUMÉRICO] {col}: {n_fail} valores no numéricos. Ejemplos: "
                      f"{non_null[numeric.isna()].unique()[:5].tolist()}")
            negatives = (numeric < 0).sum()
            if negatives > 0:
                found_issue = True
                print(f"  [NUMÉRICO] {col}: {negatives} valores negativos (revisar si es válido para el negocio)")

        # Texto / categórico
        elif not lc.endswith("_id") and lc != "id":
            non_null_str = non_null.astype(str)
            has_ws = (non_null_str.str.strip() != non_null_str).sum()
            n_unique = non_null_str.nunique()
            n_unique_lower = non_null_str.str.lower().str.strip().nunique()
            if has_ws > 0:
                found_issue = True
                print(f"  [TEXTO] {col}: {has_ws} valores con espacios extra al inicio/final")
            if n_unique_lower < n_unique and n_unique < 100:
                found_issue = True
                print(f"  [TEXTO] {col}: {n_unique} valores únicos, pero solo {n_unique_lower} "
                      f"al normalizar mayúsculas/espacios -> hay variantes del mismo valor")
    if not found_issue:
        print("  Sin inconsistencias de formato detectadas.")


### Aplicar la auditoría a las tablas de `university`

In [36]:
for name, df in [("semesters", semesters), ("professors", professors), ("students", students),
                  ("courses", courses), ("enrollments", enrollments), ("grades", grades)]:
    audit_column_types(df, name)



=== semesters ===
  Sin inconsistencias de formato detectadas.

=== professors ===
  Sin inconsistencias de formato detectadas.

=== students ===
  Sin inconsistencias de formato detectadas.

=== courses ===
  Sin inconsistencias de formato detectadas.

=== enrollments ===
  Sin inconsistencias de formato detectadas.

=== grades ===
  [NUMÉRICO] grade_id: 60000 valores no numéricos. Ejemplos: ['GRD-00000001', 'GRD-00000002', 'GRD-00000003', 'GRD-00000004', 'GRD-00000005']


### Aplicar la auditoría a las tablas de `billing`

In [37]:
for name, df in [("customers", customers), ("products", products), ("subscriptions", subscriptions),
                  ("invoices", invoices), ("invoice_items", invoice_items), ("payments", payments)]:
    audit_column_types(df, name)



=== customers ===
  Sin inconsistencias de formato detectadas.

=== products ===
  Sin inconsistencias de formato detectadas.

=== subscriptions ===
  Sin inconsistencias de formato detectadas.

=== invoices ===
  Sin inconsistencias de formato detectadas.

=== invoice_items ===
  Sin inconsistencias de formato detectadas.

=== payments ===
  Sin inconsistencias de formato detectadas.


### Aplicar la auditoría a las tablas de `crm`

In [38]:
for name, df in [("accounts", accounts), ("contacts", contacts), ("leads", leads),
                  ("opportunities", opportunities), ("opportunity_contacts", opportunity_contacts),
                  ("activities", activities)]:
    audit_column_types(df, name)



=== accounts ===
  Sin inconsistencias de formato detectadas.

=== contacts ===
  Sin inconsistencias de formato detectadas.

=== leads ===
  Sin inconsistencias de formato detectadas.

=== opportunities ===
  Sin inconsistencias de formato detectadas.

=== opportunity_contacts ===
  Sin inconsistencias de formato detectadas.

=== activities ===
  Sin inconsistencias de formato detectadas.


### Hallazgos — Auditoría de tipos y formatos

- Fechas con formatos mixtos:
- Montos/números no convertibles:
- Valores negativos donde no deberían existir:
- Texto con espacios extra o variantes de mayúsculas/minúsculas:

---
## Validación de reglas de negocio


In [39]:
def check_rule(df, condition_mask, table_name, rule_description, id_cols):
    """Aplica una regla de negocio (boolean mask) y reporta cuántas filas la violan.
    condition_mask debe ser True donde la fila ES VÁLIDA; se reportan las que NO cumplen."""
    invalid = df[~condition_mask]
    pct = len(invalid) / len(df) * 100 if len(df) else 0
    print(f"[{table_name}] {rule_description}")
    print(f"  -> {len(invalid)} de {len(df)} filas violan la regla ({pct:.2f}%)")
    if len(invalid) > 0:
        display(invalid[id_cols].head(5))
    return invalid


### `crm.opportunities` — cierre no puede ser una fecha pasada a la creación

In [40]:
opportunities["created_at_p"] = pd.to_datetime(opportunities["created_at"])
opportunities["close_date_p"] = pd.to_datetime(opportunities["close_date"])

mask = opportunities["close_date_p"] >= opportunities["created_at_p"]
invalid_opps = check_rule(
    opportunities, mask, "opportunities",
    "close_date debe ser >= created_at",
    ["opportunity_id", "created_at", "close_date"]
)


[opportunities] close_date debe ser >= created_at
  -> 1029 de 3000 filas violan la regla (34.30%)


,opportunity_id,created_at,close_date
1,OPP-0000002,2025-05-25 03:05:15,2023-09-28
4,OPP-0000005,2024-11-30 06:25:22,2023-08-30
8,OPP-0000009,2024-07-08 09:42:27,2023-08-29
11,OPP-0000012,2025-03-09 20:57:04,2023-03-03
12,OPP-0000013,2023-10-06 12:46:38,2023-06-03


### `crm.activities` — actividad no puede ocurrir antes de crear la oportunidad relacionada

In [44]:
activities_merged = activities.merge(
    opportunities[["opportunity_id", "created_at"]],
    on="opportunity_id", how="left"
)
activities_merged["occurred_at_p"] = pd.to_datetime(activities_merged["occurred_at"])

# Solo evaluamos filas donde sí hay opportunity_id (recordar: hay nulos ahí, ya detectados antes)
has_opp = activities_merged["created_at"].notna()
mask = ~has_opp | (activities_merged["occurred_at_p"] >= activities_merged["created_at"])
invalid_act = check_rule(
    activities_merged, mask, "activities",
    "occurred_at debe ser >= created_at de la oportunidad relacionada",
    ["activity_id", "occurred_at", "opportunity_id", "created_at"]
)


[activities] occurred_at debe ser >= created_at de la oportunidad relacionada
  -> 3792 de 20000 filas violan la regla (18.96%)


,activity_id,occurred_at,opportunity_id,created_at
8,ACT-00000009,2023-04-09 08:30:08,OPP-0002492,2024-03-03 09:43:18
13,ACT-00000014,2023-11-26 03:50:33,OPP-0002064,2025-02-02 14:54:02
14,ACT-00000015,2024-12-03 07:26:03,OPP-0002647,2025-09-17 22:48:35
16,ACT-00000017,2023-11-27 15:50:11,OPP-0001735,2024-12-25 21:06:07
18,ACT-00000019,2023-05-20 04:52:00,OPP-0002531,2024-05-26 22:29:58


### `billing.invoices` — vencimiento no puede ser antes de la emisión

In [45]:
invoices["issued_at_p"] = pd.to_datetime(invoices["issued_at"])
invoices["due_at_p"] = pd.to_datetime(invoices["due_at"])

mask = invoices["due_at_p"] >= invoices["issued_at_p"]
invalid_inv = check_rule(
    invoices, mask, "invoices",
    "due_at debe ser >= issued_at",
    ["invoice_id", "issued_at", "due_at"]
)


[invoices] due_at debe ser >= issued_at
  -> 0 de 50000 filas violan la regla (0.00%)


### `billing.payments` — el pago no puede ocurrir antes de emitir la factura

In [46]:
payments_merged = payments.merge(
    invoices[["invoice_id", "issued_at_p"]],
    on="invoice_id", how="left"
)
payments_merged["paid_at_p"] = pd.to_datetime(payments_merged["paid_at"])

mask = payments_merged["paid_at_p"] >= payments_merged["issued_at_p"]
invalid_pay = check_rule(
    payments_merged, mask, "payments",
    "paid_at debe ser >= issued_at de la factura relacionada",
    ["payment_id", "paid_at", "invoice_id", "issued_at_p"]
)


[payments] paid_at debe ser >= issued_at de la factura relacionada
  -> 0 de 80000 filas violan la regla (0.00%)


### `billing.subscriptions` — la suscripción no puede terminar antes de empezar

In [47]:
subscriptions["start_date_p"] = pd.to_datetime(subscriptions["start_date"])
subscriptions["end_date_p"] = pd.to_datetime(subscriptions["end_date"], errors="coerce")

# end_date puede ser nulo (suscripción activa/sin terminar) -- eso NO es un error
has_end = subscriptions["end_date_p"].notna()
mask = ~has_end | (subscriptions["end_date_p"] >= subscriptions["start_date_p"])
invalid_subs = check_rule(
    subscriptions, mask, "subscriptions",
    "end_date debe ser >= start_date (cuando end_date existe)",
    ["subscription_id", "start_date", "end_date"]
)


[subscriptions] end_date debe ser >= start_date (cuando end_date existe)
  -> 783 de 15000 filas violan la regla (5.22%)


,subscription_id,start_date,end_date
28,SUB-0000029,2024-12-03,2024-02-19
43,SUB-0000044,2024-07-16,2024-07-06
55,SUB-0000056,2024-05-27,2024-05-04
61,SUB-0000062,2025-05-20,2024-04-12
69,SUB-0000070,2025-04-16,2024-07-09


### `university.students` — edad razonable al momento de inscribirse

In [48]:
students["birth_date_p"] = pd.to_datetime(students["birth_date"])
students["enrolled_at_p"] = pd.to_datetime(students["enrolled_at"])

age_at_enroll = (students["enrolled_at_p"] - students["birth_date_p"]).dt.days / 365.25

# Rango razonable para un estudiante: 15 a 90 años. Ajusta si el negocio real difiere.
mask = age_at_enroll.between(15, 90)
students["_age_at_enroll"] = age_at_enroll
invalid_students = check_rule(
    students, mask, "students",
    "edad al inscribirse debe estar entre 15 y 90 años",
    ["student_id", "birth_date", "enrolled_at", "_age_at_enroll"]
)


[students] edad al inscribirse debe estar entre 15 y 90 años
  -> 636 de 5000 filas violan la regla (12.72%)


,student_id,birth_date,enrolled_at,_age_at_enroll
8,STU-0000009,2005-08-04,2019-01-04,13.418207
11,STU-0000012,2006-07-03,2018-03-27,11.731691
25,STU-0000026,2005-08-21,2019-04-16,13.650924
26,STU-0000027,2007-01-29,2021-03-04,14.094456
29,STU-0000030,2004-03-22,2019-03-10,14.965092


### `university.grades` — la nota no puede registrarse antes de la inscripción

In [49]:
grades_merged = grades.merge(
    enrollments[["enrollment_id", "enrolled_at"]].rename(columns={"enrolled_at": "enr_enrolled_at"}),
    on="enrollment_id", how="left"
)
grades_merged["graded_at_p"] = pd.to_datetime(grades_merged["graded_at"])
grades_merged["enr_enrolled_at_p"] = pd.to_datetime(grades_merged["enr_enrolled_at"])

mask = grades_merged["graded_at_p"] >= grades_merged["enr_enrolled_at_p"]
invalid_grades = check_rule(
    grades_merged, mask, "grades",
    "graded_at debe ser >= enrolled_at de la inscripción relacionada",
    ["grade_id", "graded_at", "enrollment_id", "enr_enrolled_at"]
)


[grades] graded_at debe ser >= enrolled_at de la inscripción relacionada
  -> 29241 de 60000 filas violan la regla (48.73%)


,grade_id,graded_at,enrollment_id,enr_enrolled_at
0,GRD-00000001,2024-10-15,ENR-00002100,2024-11-24
4,GRD-00000005,2024-07-11,ENR-00019216,2024-08-21
5,GRD-00000006,2023-03-17,ENR-00014503,2024-04-08
6,GRD-00000007,2023-08-04,ENR-00003147,2025-02-10
7,GRD-00000008,2023-05-29,ENR-00010957,2023-09-05


### Hallazgos — Reglas de negocio

| Regla | Tabla | Filas afectadas | % | Decisión propuesta para Silver |
|---|---|---|---|---|
| close_date >= created_at | opportunities | 34.30% | | |
| occurred_at >= created_at (oportunidad) | activities | 18.96% | | |
| due_at >= issued_at | invoices | 0.00% | | |
| paid_at >= issued_at | payments | 0.00% | | |
| end_date >= start_date | subscriptions | 5.22% | | |
| edad 15-90 al inscribirse | students | 12.72% |  | |
| graded_at >= enrolled_at | grades | 48.73% | | |
